In [1]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.applications.efficientnet import preprocess_input

I0000 00:00:1785494956.594442   16472 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785494957.471700   16472 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785494960.439839   16472 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [67]:
from tensorflow.keras.models import load_model

best_model = load_model("models/efficientnet_final.keras")

In [68]:
class_names = [

    "Actinic Keratoses",

    "Basal Cell Carcinoma",

    "Benign Keratosis",

    "Dermatofibroma",

    "Melanoma",

    "Melanocytic Nevus",

    "Vascular Lesion"

]

In [69]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path ="/home/aximsoft/PycharmProjects/Weekend-Task/SkinCancer_Disease/dataset/ham10000_images_part_1/ISIC_0024317.jpg"

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(
    img_array,
    axis=0
)

img_array = preprocess_input(img_array)

prediction = best_model.predict(img_array)
predicted_class = np.argmax(prediction)
print("Predicted Disease :", class_names[predicted_class])
print("Confidence :", np.max(prediction))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted Disease : Melanoma
Confidence : 0.20746876


In [70]:
for layer in best_model.layers:
    print(layer.name, layer.__class__.__name__)

efficientnetb0 Functional
global_average_pooling2d_2 GlobalAveragePooling2D
dense_4 Dense
dropout_2 Dropout
dense_5 Dense


In [71]:
import tensorflow as tf

dummy = tf.random.normal((1, 224, 224, 3))

_ = best_model(dummy)

In [72]:
print(type(best_model))

<class 'keras.src.models.sequential.Sequential'>


In [73]:
best_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,027,962 (22.99 MB)

 Trainable params: 659,463 (2.52 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 1,318,928 (5.03 MB)

In [74]:
base_model = best_model.layers[0]

print(base_model.name)

efficientnetb0


In [75]:
for layer in reversed(base_model.layers):

    if isinstance(layer, tf.keras.layers.Conv2D):

        print(layer.name)

        break

top_conv


In [76]:
last_conv_layer_name = "top_conv"

In [79]:
import tensorflow as tf

dummy = tf.zeros((1,224,224,3))

_ = best_model(dummy)

In [80]:
print(best_model.inputs)
print(best_model.outputs)

[<KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=input_layer_5>]
[<KerasTensor shape=(None, 7), dtype=float32, sparse=False, ragged=False, name=keras_tensor_5948>]


In [81]:
best_model.outputs[0]

<KerasTensor shape=(None, 7), dtype=float32, sparse=False, ragged=False, name=keras_tensor_5948>

In [82]:
base_model = best_model.layers[0]

last_conv_layer_name = "top_conv"

grad_model = tf.keras.Model(

    inputs=best_model.inputs,

    outputs=[
        base_model.get_layer(last_conv_layer_name).output,
        best_model.outputs[0]
    ]
)

ValueError: Output with path `0` is not connected to `inputs`